<a href="https://colab.research.google.com/github/Jeremy26/stereo_vision_course/blob/main/DepthSeg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Breaking MonoDepth**: From Notebook to Production

In [ ]:
!wget https://stereo-vision.s3.eu-west-3.amazonaws.com/stereo_data.zip && jar xf stereo_data.zip

In [ ]:
!jar xf stereo_vision_data.zip

In [ ]:
import os
os.chdir("stereo_data")

In [ ]:
!git clone https://github.com/nianticlabs/monodepth2.git
!cd monodepth2

In [ ]:
import sys
sys.path.append('/content/stereo_data/monodepth2')

In [ ]:
import cv2

In [ ]:
input_image = cv2.imread("/content/stereo_data/left/000009.png")

input_image = cv2.cvtColor(input_image, cv2.COLOR_BGR2RGB)

input_image.shape[:2]

In [ ]:
import cv2
import numpy as np
import torch
from monodepth2.networks import DepthDecoder, ResnetEncoder
from monodepth2.utils import download_model_if_doesnt_exist
from google.colab.patches import cv2_imshow

# Download the model weights
model_name = "mono_640x192"
download_model_if_doesnt_exist(model_name)

# Load the model components
encoder_path = f"/content/stereo_data/models/{model_name}/encoder.pth"
depth_decoder_path = f"/content/stereo_data/models/{model_name}/depth.pth"

# Load the encoder
encoder = ResnetEncoder(18, False)
loaded_dict_enc = torch.load(encoder_path, map_location='cpu')
filtered_dict_enc = {k: v for k, v in loaded_dict_enc.items() if k in encoder.state_dict()}
encoder.load_state_dict(filtered_dict_enc)

# Load the depth decoder
depth_decoder = DepthDecoder(num_ch_enc=encoder.num_ch_enc, scales=range(4))
loaded_dict = torch.load(depth_decoder_path, map_location='cpu')
depth_decoder.load_state_dict(loaded_dict)

# Set to evaluation mode
encoder.eval()
depth_decoder.eval()

# Read and preprocess an image
input_image = cv2.imread("/content/stereo_data/left/000009.png")

input_image = cv2.cvtColor(input_image, cv2.COLOR_BGR2RGB)
original_height, original_width = input_image.shape[:2]
input_image = cv2.resize(input_image, (640, 192))
input_image = input_image.transpose(2, 0, 1)
input_image = np.expand_dims(input_image, 0)
input_image = torch.from_numpy(input_image).float() / 255.0

# Predict depth
with torch.no_grad():
    features = encoder(input_image)
    outputs = depth_decoder(features)

# Generate the depth map
disp = outputs[("disp", 0)]
disp_resized = torch.nn.functional.interpolate(disp, (original_height, original_width), mode="bilinear", align_corners=False)

# Saving numpy file
depth_map = disp_resized.squeeze().cpu().numpy()
np.save("depth_map.npy", depth_map)

# Visualize the result
disp_resized_np = disp_resized.squeeze().cpu().numpy()
cv2_imshow(disp_resized_np)


In [ ]:
print(disp_resized_np*255)

In [ ]:
print(np.min(disp_resized_np))

In [ ]:
plt.imshow(disp_resized_np*255, cmap="magma")
plt.show()

In [ ]:
print(np.max(np.load('depth_map.npy')))

In [ ]:
test_depth = calc_depth_map(disp_resized_np, k_left, t_left, t_right)

In [ ]:
print((test_depth))

In [ ]:
plt.imshow(test_depth, cmap="magma")